# 7 — Caching

**The concept:** inside a convergence loop the same discipline is called over and over, often with
inputs that have not changed. `@cached` stores results keyed on the arguments, so an expensive
discipline is evaluated once per distinct input rather than once per sweep.

In [1]:
import time
from smartmdao import Pipeline, cached, MemoryBackend

calls = {"n": 0}

@cached(backend=MemoryBackend())
def expensive_analysis(span: float, chord: float) -> float:
    calls["n"] += 1
    time.sleep(0.05)                 # pretend this is a solver
    return span * chord * 1.5

started = time.perf_counter()
first = expensive_analysis(span=10.0, chord=1.5)
middle = time.perf_counter()
second = expensive_analysis(span=10.0, chord=1.5)
finished = time.perf_counter()

print(f"first call:  {first:.3f}  in {middle - started:.3f}s")
print(f"second call: {second:.3f}  in {finished - middle:.4f}s")
print(f"the function actually ran {calls['n']} time(s)")

first call:  22.500  in 0.050s
second call: 22.500  in 0.0002s
the function actually ran 1 time(s)


## The trap: `@cached` functions are keyword-only

The wrapper is `def wrapper(**kwargs)`. Inside a pipeline this is invisible, because the executor
always calls `step.fn(**params)`. Calling a cached discipline **directly** with positional
arguments fails.

In [2]:
try:
    expensive_analysis(10.0, 1.5)
except TypeError as error:
    print(f"TypeError: {error}")

print()
print("keywords work:", expensive_analysis(span=10.0, chord=1.5))

TypeError: expensive_analysis() takes 0 positional arguments but 2 were given

keywords work: 22.5


## Inside a pipeline

This is where it pays. The loop below re-evaluates its cyclic block many times, but the expensive
step's inputs stop changing early.

In [3]:
from smartmdao import HybridSolver

aero_calls = {"n": 0}

@cached(backend=MemoryBackend())
def aerodynamics(span: float) -> float:
    aero_calls["n"] += 1
    return 0.9 * span

loop = Pipeline(solver=HybridSolver(max_iterations=50))
loop.add(aerodynamics, outputs=["lift_coeff"])

@loop.step(outputs=["mass"])
def structure(lift_coeff: float, mass: float) -> float:
    return 100.0 + 0.5 * lift_coeff + 0.05 * mass

out = loop.run(span=10.0, mass=0.0)
print(f"mass -> {out['mass']:.6f}")
print(f"aerodynamics evaluated {aero_calls['n']} time(s) despite "
      f"{out['convergence_reports'][0].iterations} iterations")

mass -> 110.000000
aerodynamics evaluated 1 time(s) despite 8 iterations


## The four backends

| Backend | Stores | Use for |
|---|---|---|
| `MemoryBackend` | a dict, process-lifetime | the default; fastest |
| `HistoryBackend` | every call, in order | auditing what was evaluated |
| `PickleDiskBackend` | pickled files | arbitrary Python objects, across runs |
| `HDF5Backend` | an HDF5 file | scalars, strings and numpy arrays, across runs |

In [4]:
from smartmdao import HistoryBackend

history = HistoryBackend()

@cached(backend=history)
def sampled(x: float) -> float:
    return x ** 2

for value in (1.0, 2.0, 1.0, 3.0):
    sampled(x=value)

print("distinct entries stored:", len(history.store))
print("call history:", history.history)

distinct entries stored: 3
call history: defaultdict(<class 'list'>, {'sampled': [1.0, 4.0, 9.0]})


In [5]:
import tempfile, pathlib
from smartmdao import PickleDiskBackend

directory = pathlib.Path(tempfile.mkdtemp())
disk_calls = {"n": 0}

@cached(backend=PickleDiskBackend(directory=str(directory)))
def structural_model(load: float) -> dict:
    disk_calls["n"] += 1
    return {"stress": load * 2.5, "margin": 1.4}

print(structural_model(load=100.0))
print(structural_model(load=100.0))
print(f"evaluated {disk_calls['n']} time(s); "
      f"{len(list(directory.glob('*')))} file(s) on disk")

{'stress': 250.0, 'margin': 1.4}
{'stress': 250.0, 'margin': 1.4}
evaluated 1 time(s); 1 file(s) on disk


`HDF5Backend` is the one to reach for when results are numeric arrays and you want them
readable by other tools. It **cannot store arbitrary Python objects** — scalars, strings and numpy
arrays only — and the failure happens at write time, which is an unpleasant place to discover it.

In [6]:
import numpy as np
from smartmdao import HDF5Backend

h5_path = directory / "cache.h5"

@cached(backend=HDF5Backend(filepath=str(h5_path)))
def pressure_field(alpha: float) -> np.ndarray:
    return np.linspace(0.0, alpha, 5)

print(pressure_field(alpha=2.0))
print(pressure_field(alpha=2.0))
print("file written:", h5_path.exists())

@cached(backend=HDF5Backend(filepath=str(directory / "bad.h5")))
def returns_a_dict(x: float) -> dict:
    return {"not": "storable"}

try:
    returns_a_dict(x=1.0)
except Exception as error:
    print(f"{type(error).__name__}: {str(error)[:80]}")

[0.  0.5 1.  1.5 2. ]
[0.  0.5 1.  1.5 2. ]
file written: True
TypeError: Object dtype dtype('O') has no native HDF5 equivalent


---

**Next:** [8 — Optimization](08-optimization.ipynb).